# CS467 Advanced Artificial Intelligence - Mini-Project (MP-C15)
## Study Resource and Exam Prep Material Recommender
- **Student Name**: Chu Văn Nhật Thanh
- **Student ID**: 29211125861
- **Class / Section**: CS467C (No. 15)
- **Instructor**: Puri Vikram
- **Domain**: Student Learning Hub
---
This notebook walks through the end-to-end implementation of the study resource recommender combining **Natural Language Processing (NLP)** and **Hybrid Recommendation Systems**.

### 1. Imports and Dataset Loading

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

items_df = pd.read_csv('../data/items.csv')
ratings_df = pd.read_csv('../data/ratings.csv')

print(f'Items count: {len(items_df)}')
print(f'Ratings count: {len(ratings_df)}')
items_df.head(5)

### 2. Natural Language Processing (NLP) Pipeline
- Text cleaning & stop-word removal
- Unigram + Bigram TF-IDF vectorization
- Extraction of matching study concepts

In [ ]:
from src.nlp_pipeline import StudyResourceNLPPipeline, clean_text

nlp_pipe = StudyResourceNLPPipeline().fit(items_df)
print(f'Fitted TF-IDF vocabulary size: {len(nlp_pipe.feature_names)} terms')

sample_query = 'practice problems on probability, bayes theorem, and distributions'
print('Cleaned Query:', clean_text(sample_query))
q_vec = nlp_pipe.transform_query(sample_query)
matched_concepts = nlp_pipe.extract_matching_concepts(q_vec, item_index=0)
print('Matched concepts for RES001:', matched_concepts)

### 3. Recommendation Engines Initialization
1. Popularity Baseline (with rating threshold & filtering)
2. NLP Content-Based Recommender (TF-IDF Cosine Similarity)
3. Item-Based Collaborative Filtering (IBCF)
4. Hybrid Recommender (Blends CF and NLP)

In [ ]:
from src.recommender import (
    PopularityBaselineRecommender,
    NLPContentBasedRecommender,
    ItemBasedCollaborativeFiltering,
    HybridRecommender
)

pop_rec = PopularityBaselineRecommender(min_ratings_threshold=2).fit(items_df, ratings_df)
nlp_rec = NLPContentBasedRecommender(nlp_pipe).fit(items_df, ratings_df)
ibcf_rec = ItemBasedCollaborativeFiltering().fit(ratings_df)
hybrid_rec = HybridRecommender(nlp_rec, ibcf_rec, alpha=0.5).fit(items_df, ratings_df)
print('All recommendation models initialized and trained successfully!')

### 4. Mandatory Test Cases Execution
- **Test Case 1**: Weak-topic query recommendation
- **Test Case 2**: Related-material follow-up recommendations
- **Test Case 3**: Cold-start test with a zero-rating cheatsheet

In [ ]:
print('--- TEST CASE 1: Weak-Topic Query ---')
tc1_recs = nlp_rec.recommend_by_query('practice problems on probability, bayes theorem, and distributions', top_k=5)
for r in tc1_recs:
    print(f"[{r['item_id']}] {r['resource_title']} | Score: {r['score']:.4f} | Concepts: {', '.join(r['matching_concepts'])}")

print('\n--- TEST CASE 2: Related-Material Follow-up (Seed: RES001) ---')
tc2_recs = nlp_rec.recommend_by_item('RES001', top_k=5)
for r in tc2_recs:
    print(f"[{r['item_id']}] {r['resource_title']} | Score: {r['score']:.4f} | Shared: {', '.join(r['matching_concepts'])}")

print('\n--- TEST CASE 3: Cold-Start Test (Zero-Rating Cheatsheet) ---')
new_cheatsheet = {
    'item_id': 'RES041_NEW',
    'resource_title': 'Continuous and Discrete Fourier Transform Exam Cheatsheet',
    'subject': 'Applied Mathematics',
    'resource_type': 'Cheatsheet',
    'summary_text': 'Formula sheet for Fourier transform, FFT, frequency filtering.',
    'concept_tags': 'fourier transform; fft; signal processing; frequency filtering'
}
ext_items = pd.concat([items_df, pd.DataFrame([new_cheatsheet])], ignore_index=True)
ext_nlp = StudyResourceNLPPipeline().fit(ext_items)
ext_rec = NLPContentBasedRecommender(ext_nlp).fit(ext_items, ratings_df)
cold_recs = ext_rec.recommend_by_query('fourier transform frequency filtering cheatsheet', top_k=3)
for r in cold_recs:
    print(f"[{r['item_id']}] {r['resource_title']} | Score: {r['score']:.4f}")

### 5. Offline Evaluation & Visualizations

In [ ]:
from src.evaluate import train_test_split_by_user, evaluate_offline

train_df, test_df = train_test_split_by_user(ratings_df, test_ratio=0.2, random_state=42)
results = evaluate_offline(items_df, train_df, test_df)

res_df = pd.DataFrame(results).T
print(res_df)